# Bank of Baroda — Intraday Stock Analysis

A cleaned and portable version of the original Hackveda candlestick task.

**Dataset:** 5-minute OHLCV observations for Bank of Baroda  
**Coverage in supplied data:** 06 Mar 2023 – 11 Apr 2023

This notebook focuses on data preparation, validation, descriptive statistics, candlestick visualization, moving averages, volume, and intraday returns. It is an analytical exercise, not a trading or forecasting model.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

DATA_PATH = Path("../data/Bank_of_Baroda_5min_OHLCV.csv")
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df.head()

In [ ]:
required = ["Date", "Open", "High", "Low", "Close", "Volume"]
assert set(required).issubset(df.columns)

df = df[required].copy()
for col in required[1:]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.dropna(subset=required).drop_duplicates("Date").sort_values("Date")
valid_ohlc = (
    (df["High"] >= df[["Open","Low","Close"]].max(axis=1)) &
    (df["Low"] <= df[["Open","High","Close"]].min(axis=1)) &
    (df["High"] >= df["Low"]) &
    (df[["Open","High","Low","Close"]] > 0).all(axis=1) &
    (df["Volume"] >= 0)
)
df = df.loc[valid_ohlc].copy()

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {len(df):,}")
print(f"Date range: {df['Date'].min()} → {df['Date'].max()}")

In [ ]:
df["Bar Return %"] = df.groupby(df["Date"].dt.date)["Close"].pct_change() * 100
df["MA 20"] = df["Close"].rolling(20, min_periods=1).mean()
df["MA 50"] = df["Close"].rolling(50, min_periods=1).mean()
df["Intraday Range %"] = (df["High"] - df["Low"]) / df["Open"] * 100

summary = {
    "Minimum close": df["Close"].min(),
    "Maximum close": df["Close"].max(),
    "Average close": df["Close"].mean(),
    "Average 5-minute volume": df["Volume"].mean(),
    "Trading sessions": df["Date"].dt.date.nunique(),
}
pd.Series(summary)

In [ ]:
fig = go.Figure(go.Candlestick(
    x=df["Date"], open=df["Open"], high=df["High"],
    low=df["Low"], close=df["Close"], name="Bank of Baroda"
))
fig.update_layout(
    title="Bank of Baroda — 5-Minute OHLC Candlestick",
    xaxis_title="Date & Time",
    yaxis_title="Price (₹)",
    template="plotly_dark",
    xaxis_rangeslider_visible=False,
    height=600
)
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df["Date"], y=df["Close"], name="Close", mode="lines"))
fig.add_trace(go.Scatter(x=df["Date"], y=df["MA 20"], name="20-bar MA", mode="lines"))
fig.add_trace(go.Scatter(x=df["Date"], y=df["MA 50"], name="50-bar MA", mode="lines"))
fig.update_layout(
    title="Bank of Baroda — Closing Price and Rolling Averages",
    yaxis_title="Price (₹)", template="plotly_dark", height=500
)
fig.show()

In [ ]:
session = (
    df.assign(Session=df["Date"].dt.date)
      .groupby("Session")
      .agg(Open=("Open","first"), High=("High","max"), Low=("Low","min"),
           Close=("Close","last"), Volume=("Volume","sum"), Bars=("Close","size"))
      .reset_index()
)
session["Return %"] = (session["Close"] / session["Open"] - 1) * 100
session["Range %"] = (session["High"] - session["Low"]) / session["Open"] * 100
session.head()

## Interpretation Notes

- Candlesticks show the five-minute Open, High, Low, and Close for each observation.
- Moving averages summarize historical closing-price behaviour over rolling bar windows.
- Session return is calculated from the first open to the final close of each trading date.
- These calculations describe the supplied historical dataset; they are not forecasts or investment recommendations.